# The variance premium: VIX against realised variance

**Plain idea.** The VIX is a price for the next 30 days of S&P 500 variance,
quoted as a volatility. Square it and compare it with the variance that then
happened. On average the quote is higher: buyers of crash insurance pay a
premium (Carr and Wu 2009, RFS 22(3)).

**What this notebook uses.** Synthetic data only, so it runs offline. The
real-data study is `research/vix/vix_rv.py` (CBOE published VIX level and
the French daily market return; no option prices).

In [ ]:
import _paths  # noqa: F401  (puts research/ modules on sys.path)
import numpy as np
import pandas as pd
import vix_rv

## Simulated market

Daily volatility follows a slow random walk in logs. The "VIX" quotes the
true volatility plus a fixed premium of 3 vol points, so we know the answer.

In [ ]:
rng = np.random.default_rng(2026)
days = pd.bdate_range("2005-01-03", periods=2000)
log_vol = np.log(0.18) + np.cumsum(rng.normal(0, 0.03, len(days)))
log_vol = np.log(0.18) + 0.97 * (log_vol - np.log(0.18))
vol = pd.Series(np.exp(log_vol), index=days)
ret = pd.Series(rng.normal(0, 1, len(days)) * vol / np.sqrt(252), index=days)
vix = 100 * (vol + 0.03)

## Task 1: forward realised variance by hand

Compute `rv_hand`: for each day, `252/21` times the sum of the squared
returns over the **next** 21 days (not including today). Use a plain loop.

In [ ]:
r2 = ret.to_numpy() ** 2
rv_hand = np.full(len(r2), np.nan)
for t in range(len(r2) - 21):
    rv_hand[t] = 252 / 21 * r2[t + 1:t + 22].sum()

In [ ]:
lib = vix_rv.forward_realised_variance(ret).to_numpy()
assert np.allclose(rv_hand, lib, equal_nan=True, rtol=1e-12, atol=0)

## Task 2: the study

Run `vix_rv.vix_study(vix, ret)` into `res` and print the mean premium in
variance units and the Mincer-Zarnowitz slope. The lock passes because all
dates are before 2016.

In [ ]:
res = vix_rv.vix_study(vix, ret)
print(f"mean VRP {res['mean_vrp']:.4f}  NW t {res['nw_t_vrp']:.1f}  "
      f"MZ slope {res['mincer_zarnowitz']['slope']:.2f}")

In [ ]:
assert res["mean_vrp"] > 0
assert res["nw_t_vrp"] > 2
assert "no option prices" in res["label"]

**What to take away.** The premium is positive by construction here. On real
data it is positive on average but with large negative spikes when realised
variance jumps; that tail is what the sellers are paid for.